# AIH — BreaKHis 400× (Tiers 1, 2, 3)

**Colab notebook.** Code from GitHub, dataset from the existing `aih.tar.gz` on Drive, and **all outputs written directly to Drive via a `results/` symlink** — so if the runtime dies, completed seeds and logs survive.

In [1]:
# Cell 1 — GPU/CPU
!nvidia-smi

Thu May 14 08:07:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# Cell 2 — mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Cell 3 — get the latest code from GitHub
import os, subprocess

REPO_URL = "https://github.com/sathish-kumaresan/aih.git"
REPO_DIR = "/content/aih"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    print("Cloning fresh...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

%cd /content/aih
!git log -1 --oneline

Cloning fresh...
/content/aih
6b4cb41 (HEAD -> main, origin/main, origin/HEAD) Added the run all script + fixed minor bugs


In [5]:
# Cell 4 — extract ONLY the data/ directory from the tarball.
import os

TARBALL_DRIVE = "/content/drive/MyDrive/Colab Notebooks/aih.tar.gz"
TARBALL_LOCAL = "/content/aih.tar.gz"
DATA_MARKER = "/content/aih/data/raw/400X/test/malignant"

# Skip if data already fully extracted
if os.path.isdir(DATA_MARKER) and len(os.listdir(DATA_MARKER)) > 100:
    print("Data already extracted at /content/aih/data/raw/400X — skipping.")
else:
    print(f"Copying tarball off Drive ({TARBALL_DRIVE}) ...")
    !cp "{TARBALL_DRIVE}" "{TARBALL_LOCAL}"
    !ls -lh "{TARBALL_LOCAL}"

    print(f"\nExtracting ./data from local tarball ...")
    !tar -xzf "{TARBALL_LOCAL}" -C /content/aih ./data

    print("\nCleaning up local tarball copy ...")
    !rm "{TARBALL_LOCAL}"

# Verify counts (expected: 371 / 777 / 176 / 369)
print("\ntrain/benign    :")
!ls /content/aih/data/raw/400X/train/benign | wc -l
print("train/malignant :")
!ls /content/aih/data/raw/400X/train/malignant | wc -l
print("test/benign     :")
!ls /content/aih/data/raw/400X/test/benign | wc -l
print("test/malignant  :")
!ls /content/aih/data/raw/400X/test/malignant | wc -l

Copying tarball off Drive (/content/drive/MyDrive/Colab Notebooks/aih.tar.gz) ...
-rw------- 1 root root 803M May 14 08:09 /content/aih.tar.gz

Extracting ./data from local tarball ...

Cleaning up local tarball copy ...

train/benign    :
371
train/malignant :
777
test/benign     :
176
test/malignant  :
369


In [6]:
# Cell 5 — Point results/ at a folder on Drive. If the runtime dies, completed work is already saved.
import os

DRIVE_RESULTS = "/content/drive/MyDrive/aih_results"
LOCAL_RESULTS = "/content/aih/results"

os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.makedirs(os.path.join(DRIVE_RESULTS, "logs"), exist_ok=True)

# Remove any existing local results/ and re-link to Drive
if os.path.islink(LOCAL_RESULTS) or os.path.exists(LOCAL_RESULTS):
    !rm -rf "{LOCAL_RESULTS}"
!ln -s "{DRIVE_RESULTS}" "{LOCAL_RESULTS}"

!ls -la /content/aih/results
print()
print("Contents already on Drive (from any prior runs):")
!ls -la "{DRIVE_RESULTS}"

lrwxrwxrwx 1 root root 34 May 14 08:09 /content/aih/results -> /content/drive/MyDrive/aih_results

Contents already on Drive (from any prior runs):
total 4
drwx------ 2 root root 4096 May 14 07:41 logs


In [7]:
# Cell 6 — install required packages.
%cd /content/aih
!pip install -q -r requirements.txt

/content/aih
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 103.9 MB/s eta 0:00:00


In [8]:
# Cell 7 — sanity check.
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 native:", torch.cuda.is_bf16_supported())
else:
    print("(CPU runtime — Tier 1 will run fine, Tiers 2+3 will fail until you switch to a GPU runtime)")

# Backbone load test only if GPU is present
if torch.cuda.is_available():
    import timm
    m = timm.create_model('tf_efficientnetv2_s.in21k_ft_in1k',
                          pretrained=True, num_classes=0, global_pool='').cuda().eval()
    out = m(torch.zeros(1, 3, 384, 384).cuda())
    print("Backbone output shape:", tuple(out.shape))
    del m
    torch.cuda.empty_cache()

print("\n--- What's already on Drive from prior runs ---")
!ls -la /content/aih/results/ 2>/dev/null
print("\n--- Reports so far ---")
!ls -la /content/aih/results/reports/ 2>/dev/null || echo "(no reports yet)"
print("\n--- Logs so far ---")
!ls -la /content/aih/results/logs/ 2>/dev/null || echo "(no logs yet)"

torch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
bf16 native: True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

Backbone output shape: (1, 1280, 12, 12)

--- What's already on Drive from prior runs ---
total 4
drwx------ 2 root root 4096 May 14 07:41 logs

--- Reports so far ---
(no reports yet)

--- Logs so far ---
total 0


In [9]:
# Cell 8 — keepalive (optional)
from IPython.display import Javascript, display
display(Javascript('''
function ColabKeepAlive(){
  console.log("keepalive ping");
  const btn = document.querySelector("colab-toolbar-button#connect");
  if (btn) { btn.click(); }
}
setInterval(ColabKeepAlive, 60000);
'''))
print("Keepalive registered (pings every 60 s). Don't close this tab.")

<IPython.core.display.Javascript object>

Keepalive registered (pings every 60 s). Don't close this tab.


## Training cells

Each tier writes its outputs to `results/` (= Drive).

In [10]:
# Cell 9 — Tier 1 (PFTAS + SVM).
%cd /content/aih
!stdbuf -oL python -u train.py --tier 1 2>&1 | stdbuf -oL tee results/logs/tier1.log

print("\n--- Tier 1 outputs on Drive ---")
!ls -la results/reports/ 2>/dev/null | grep tier1 || echo "(no tier1 reports found)"

/content/aih
on-disk images: 1693
on-disk patients: 81
leaked patients across provided train/test: 80
PFTAS: 100%|██████████| 1693/1693 [07:26<00:00,  3.79it/s]
split honest: train 1333 images / 64 patients, test 360 images / 17 patients
split provided: train 1148 images / 81 patients, test 545 images / 80 patients
[seed 1] hyperparams = {'C': 10.0, 'gamma': 'scale'}
[seed 1/honest] img AUROC 0.783, pat AUROC 0.909
[seed 1/provided] img AUROC 0.909, pat AUROC 0.987
[seed 2] hyperparams = {'C': 1.0, 'gamma': 'scale'}
[seed 2/honest] img AUROC 0.731, pat AUROC 0.848
[seed 2/provided] img AUROC 0.854, pat AUROC 0.940
[seed 3] hyperparams = {'C': 10.0, 'gamma': 0.001}
[seed 3/honest] img AUROC 0.704, pat AUROC 0.773
[seed 3/provided] img AUROC 0.833, pat AUROC 0.921
[seed 4] hyperparams = {'C': 1.0, 'gamma': 0.001}
[seed 4/honest] img AUROC 0.626, pat AUROC 0.636
[seed 4/provided] img AUROC 0.769, pat AUROC 0.857
[seed 5] hyperparams = {'C': 1.0, 'gamma': 0.01}
[seed 5/honest] img AUROC 0.

In [11]:
# Cell 10 — Tier 2 (EfficientNetV2-S)
%cd /content/aih
!stdbuf -oL python -u train.py --tier 2 2>&1 | stdbuf -oL tee results/logs/tier2.log

print("\n--- Tier 2 outputs on Drive ---")
!ls -la results/reports/ 2>/dev/null | grep tier2 || echo "(no tier2 reports found)"

/content/aih
GPU: NVIDIA A100-SXM4-40GB  CUDA: 12.8  torch: 2.10.0+cu128
bf16 supported: True
backbone output shape: (1, 1280, 12, 12)
on-disk images: 1693
on-disk patients: 81
leaked patients across provided train/test: 80
val: 190 images / 10 patients  present in provided_train: 10  in provided_test: 10  (early-stopping leak into provided protocol)
split honest: train 1143 images / 54 patients, test 360 images / 17 patients
split provided: train 1022 images / 71 patients, test 545 images / 80 patients
[seed 1] hyperparams = {}
  epoch 0: val_auroc=0.9536  (best)
  epoch 1: val_auroc=0.9705  (best)
  epoch 2: val_auroc=0.9653  (patience 1/3)
  epoch 3: val_auroc=0.9727  (best)
  epoch 4: val_auroc=0.9713  (patience 1/3)
  epoch 5: val_auroc=0.9656  (patience 2/3)
  epoch 6: val_auroc=0.9691  (patience 3/3)
  early-stop at epoch 6
[seed 1/honest] img AUROC 0.883, pat AUROC 0.939
  epoch 0: val_auroc=0.9808  (best)
  epoch 1: val_auroc=0.9857  (best)
  epoch 2: val_auroc=0.9914  (best)


In [12]:
# Cell 11 — Tier 3 (EfficientNetV2-S + CBAM).
%cd /content/aih
!stdbuf -oL python -u train.py --tier 3 2>&1 | stdbuf -oL tee results/logs/tier3.log

print("\n--- Tier 3 outputs on Drive ---")
!ls -la results/reports/ 2>/dev/null | grep tier3 || echo "(no tier3 reports found)"

/content/aih
GPU: NVIDIA A100-SXM4-40GB  CUDA: 12.8  torch: 2.10.0+cu128
bf16 supported: True
backbone output shape: (1, 1280, 12, 12)
on-disk images: 1693
on-disk patients: 81
leaked patients across provided train/test: 80
val: 190 images / 10 patients  present in provided_train: 10  in provided_test: 10  (early-stopping leak into provided protocol)
split honest: train 1143 images / 54 patients, test 360 images / 17 patients
split provided: train 1022 images / 71 patients, test 545 images / 80 patients
[seed 1] hyperparams = {}
  epoch 0: val_auroc=0.9104  (best)
  epoch 1: val_auroc=0.9583  (best)
  epoch 2: val_auroc=0.9451  (patience 1/3)
  epoch 3: val_auroc=0.9566  (patience 2/3)
  epoch 4: val_auroc=0.9588  (best)
  epoch 5: val_auroc=0.9710  (best)
  epoch 6: val_auroc=0.9630  (patience 1/3)
  epoch 7: val_auroc=0.9541  (patience 2/3)
  epoch 8: val_auroc=0.9751  (best)
  epoch 9: val_auroc=0.9456  (patience 1/3)
  epoch 10: val_auroc=0.9571  (patience 2/3)
  epoch 11: val_auro

In [13]:
# Cell 12 — final summary: list everything in MyDrive/aih_results/
import os

DRIVE_RESULTS = "/content/drive/MyDrive/aih_results"

print("=" * 60)
print(f"Contents of {DRIVE_RESULTS}")
print("=" * 60)
for root, dirs, files in os.walk(DRIVE_RESULTS):
    rel = os.path.relpath(root, DRIVE_RESULTS)
    depth = 0 if rel == "." else rel.count(os.sep) + 1
    indent = "  " * depth
    folder = "." if rel == "." else os.path.basename(root)
    print(f"{indent}{folder}/")
    for f in sorted(files):
        path = os.path.join(root, f)
        try:
            size_mb = os.path.getsize(path) / (1024 * 1024)
            print(f"{indent}  {f}  ({size_mb:.2f} MB)")
        except OSError:
            print(f"{indent}  {f}  (size unknown)")

Contents of /content/drive/MyDrive/aih_results
./
  logs/
    tier1.log  (0.11 MB)
    tier2.log  (0.00 MB)
    tier3.log  (0.00 MB)
  runs/
    tier1/
      leak_status.json  (0.00 MB)
      honest/
        seed1/
          best_params.json  (0.00 MB)
          config_resolved.yaml  (0.00 MB)
          metrics.json  (0.00 MB)
          model.joblib  (0.68 MB)
          predictions.parquet  (0.01 MB)
        seed2/
          best_params.json  (0.00 MB)
          config_resolved.yaml  (0.00 MB)
          metrics.json  (0.00 MB)
          model.joblib  (0.87 MB)
          predictions.parquet  (0.01 MB)
        seed3/
          best_params.json  (0.00 MB)
          config_resolved.yaml  (0.00 MB)
          metrics.json  (0.00 MB)
          model.joblib  (0.81 MB)
          predictions.parquet  (0.01 MB)
        seed4/
          best_params.json  (0.00 MB)
          config_resolved.yaml  (0.00 MB)
          metrics.json  (0.00 MB)
          model.joblib  (0.91 MB)
          predictions.par